# Serving Your Portfolio Model with Flask

Flask is a lightweight synchronous web framework — battle-tested, simple, and widely deployed. In this notebook you will wrap **the model you exported from AIAT 114 or AIAT 122** in a Flask API and test it entirely inside the notebook using Flask's built-in test client (no live server needed).

## 🔗 Where this fits

**Builds on:** AIAT 115 (Course 05) — Unit 5, `08_deployment.ipynb` — where you pickled a model, wrote a `model_metadata.json` beside it, and served it from Flask and FastAPI. Course 11 makes that pairing formal: every artifact travels with a `model_card.json`, and the serving code reads the card instead of hard-coding what the model expects.

**Needs:** a portfolio model — see [`../../PORTFOLIO_MODEL.md`](../../PORTFOLIO_MODEL.md). Without one, this notebook serves the named fallback `wdbc-baseline` and says so in its output.


## Learning Objectives

By the end of this notebook you will be able to:
1. Explain when to choose Flask vs FastAPI for model serving
2. Build a Flask app that serves your own artifact via `/predict` and `/health`
3. Write input validation by hand and return correct HTTP status codes (400 / 500)
4. Test a Flask app in-notebook using `app.test_client()`
5. Verify a serving layer against the **golden batch** recorded in the model card
6. Describe how to run Flask with Gunicorn in production

## 1. Flask vs FastAPI — Quick Comparison

| | Flask | FastAPI |
|---|---|---|
| Style | Synchronous (WSGI) | Async-first (ASGI) |
| Docs | Manual | Auto OpenAPI / Swagger |
| Validation | Manual | Pydantic (automatic) |
| Maturity | ~15 years, massive ecosystem | ~5 years, fast-growing |
| Best for | Simple APIs, legacy codebases | High-throughput, data validation |

Choose Flask when your team already uses it or when you need a simpler mental model. FastAPI is covered in the next notebook.

## 2. Load the Portfolio Model

No training happens in this lesson — the model arrived finished. We load the artifact plus the card that says what it expects — feature order, class names, a real sample row, and a golden batch we will use in section 5.

In [1]:
# WHAT: load YOUR portfolio model and its card - no training happens here.
# WHY: the Flask app must load a FINISHED artifact from disk. Training offline and
# serving from an artifact is the separation every real deployment relies on, and
# in this course the offline training already happened, in AIAT 114 or AIAT 122.
import json
import sys
from pathlib import Path

for _d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_d / "portfolio_model.py").exists():
        COURSE11 = _d
        break
else:
    raise FileNotFoundError("Could not find portfolio_model.py - run this notebook from inside Course 11.")

if str(COURSE11) not in sys.path:
    sys.path.insert(0, str(COURSE11))

import portfolio_model as pf

# Builds the named fallback 'wdbc-baseline' if you have not exported a model yet.
model, card = pf.load_portfolio_model()
MODEL_DIR = pf.portfolio_dir()

# The API field names Flask will require, derived from the card's feature order.
FIELDS = pf.schema_fields(card)
FIELD_ORDER = [field for field, _ in FIELDS]
print(f"\nAPI will require {len(FIELD_ORDER)} fields, e.g. {FIELD_ORDER[:3]}")


FALLBACK MODEL 'wdbc-baseline' — this is NOT your model.
  directory   : /Users/abdullah/ai-diploma-portfolio
  artifact    : model.joblib  (sklearn)
  task        : classification  ->  2 classes ['malignant', 'benign']
  features    : 30 (first three: ['mean radius', 'mean texture', 'mean perimeter'])
  accuracy    : 0.9825 on held-out 20% (random_state=42, stratified)
  Export your own model from AIAT 114 or AIAT 122 and re-run: see Course 11/PORTFOLIO_MODEL.md

API will require 30 fields, e.g. ['mean_radius', 'mean_texture', 'mean_perimeter']


## 3. Write the Flask Application

`%%writefile` saves the cell content to a file. The key design decisions:
- Load the artifact and card **once** at startup (not inside the predict function)
- Derive the required fields from `card["feature_names"]` — never type them by hand
- Return **400** for bad input (client's fault), **500** for unexpected errors (server's fault)
- Keep the route handlers thin — delegate logic to helper functions

Watch how much code `validate_input` takes. Flask gives you nothing for free here; the next notebook deletes this entire function and replaces it with a type hint.

In [2]:
%%writefile /tmp/flask_app.py
# WHAT: write the complete Flask serving app to /tmp/flask_app.py.
# WHY: this file is the deployable unit - note the three responsibilities it
# separates: startup loading, input validation, and error-coded responses.
# Flask has no Pydantic, so every check below is one we write by hand. Keep that
# in mind when the next notebook does the same job with a type hint.
import json
import os
import re
from pathlib import Path

import numpy as np
from flask import Flask, request, jsonify

app = Flask(__name__)

# --- Load the artifact and its card once, at startup ---
MODEL_DIR = Path(os.environ.get("AI_DIPLOMA_PORTFOLIO",
                                str(Path.home() / "ai-diploma-portfolio")))
CARD = json.loads((MODEL_DIR / "model_card.json").read_text())

if CARD["framework"] == "sklearn":
    import joblib
    _model = joblib.load(MODEL_DIR / CARD["artifact"])

    def probabilities(rows):
        return np.asarray(_model.predict_proba(rows))

elif CARD["framework"] == "onnx":
    import onnxruntime as ort
    _session = ort.InferenceSession(str(MODEL_DIR / CARD["artifact"]),
                                    providers=["CPUExecutionProvider"])
    _input = _session.get_inputs()[0].name

    def probabilities(rows):
        logits = np.asarray(_session.run(None, {_input: np.asarray(rows, dtype=np.float32)})[0])
        exp = np.exp(logits - logits.max(axis=1, keepdims=True))
        return exp / exp.sum(axis=1, keepdims=True)

else:
    raise RuntimeError(f"This app serves 'sklearn' or 'onnx' artifacts, not {CARD['framework']!r}.")


def api_field(name):
    slug = re.sub(r"\W+", "_", str(name).strip().lower()).strip("_")
    return f"f_{slug}" if not slug or slug[0].isdigit() else slug

# The required request fields ARE the model's features, in training order.
REQUIRED_FEATURES = [api_field(n) for n in CARD["feature_names"]]
CLASSES = CARD["class_names"]


# One function owns ALL input checking - the endpoint stays readable.
def validate_input(data):
    """Return (features_array, error_message). error_message is None if valid."""
    if not isinstance(data, dict):
        return None, "Request body must be a JSON object"
    missing = [f for f in REQUIRED_FEATURES if f not in data]
    if missing:
        return None, f"Missing fields: {missing[:5]}{'...' if len(missing) > 5 else ''}"
    try:
        features = np.array([[float(data[f]) for f in REQUIRED_FEATURES]])
    except (TypeError, ValueError) as e:
        return None, f"All fields must be numeric: {e}"
    return features, None


# Liveness endpoint: load balancers and Kubernetes probes call this.
@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok",
                    "model": CARD["name"],
                    "source": CARD["source_course"],
                    "n_features": len(REQUIRED_FEATURES)}), 200


# The prediction endpoint: validate first, predict second, report errors honestly.
@app.route("/predict", methods=["POST"])
def predict():
    features, error = validate_input(request.get_json(silent=True))
    if error:
        # 400 = the CLIENT sent something wrong. Say exactly what.
        return jsonify({"error": error}), 400
    try:
        proba = probabilities(features)[0]
        class_id = int(np.argmax(proba))
        return jsonify({
            "prediction": CLASSES[class_id],
            "class_id": class_id,
            "confidence": round(float(proba[class_id]), 4),
        }), 200
    except Exception as e:  # noqa: BLE001 - a serving app must never leak a stack trace
        # 500 = OUR fault. Log the detail, return something safe.
        app.logger.exception("prediction failed")
        return jsonify({"error": "internal error", "detail": str(e)}), 500


Overwriting /tmp/flask_app.py


## 4. Test with Flask's Test Client

`app.test_client()` lets you send HTTP requests to your Flask app without starting a real server. This is how you write tests and also how we experiment in-notebook.

In [3]:
# WHAT: exercise the app through Flask's in-process test client - health check,
# a valid request built from the card, then two malformed requests.
# WHY: each test pins down one guarantee of the API contract; the expected status
# codes (200 vs 400) matter as much as the prediction itself.
import importlib
import sys

sys.path.insert(0, "/tmp")
import flask_app as flask_module
importlib.reload(flask_module)
client = flask_module.app.test_client()

# --- Test 1: Health check ---
resp = client.get("/health")
print("Health check :", resp.status_code, resp.get_json())

# --- Test 2: Valid prediction, from the card's real sample row ---
payload = dict(zip(FIELD_ORDER, card["sample_input"]))
resp = client.post("/predict", json=payload)
print("Valid request:", resp.status_code, resp.get_json())

# --- Test 3: Missing fields (expect 400) ---
resp = client.post("/predict", json={FIELD_ORDER[0]: payload[FIELD_ORDER[0]]})
print("Missing      :", resp.status_code, resp.get_json())

# --- Test 4: Non-numeric value (expect 400) ---
resp = client.post("/predict", json={**payload, FIELD_ORDER[0]: "big"})
print("Non-numeric  :", resp.status_code, resp.get_json())


Health check : 200 {'model': 'wdbc-baseline', 'n_features': 30, 'source': 'AIAT 125 fallback', 'status': 'ok'}
Valid request: 200 {'class_id': 1, 'confidence': 0.9991, 'prediction': 'benign'}
Missing      : 400 {'error': "Missing fields: ['mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness', 'mean_compactness']..."}
Non-numeric  : 400 {'error': "All fields must be numeric: could not convert string to float: 'big'"}


## 5. Golden-Batch Verification

The card carries `sample_batch` — up to 20 real held-out rows — together with `sample_batch_predictions`, the class the model produced for each of those rows **at export time**.

That pairing is a regression test for the *serving stack*. Push the golden rows through the API and every answer must still match. If one does not, the model file has not changed — so the fault is in your serving code: a scrambled feature order, a lost scaler, a float parsed as a string. This is the single cheapest check that stands between you and a silent production bug.

In [4]:
# WHAT: replay the card's golden batch through the HTTP API and compare each
# answer with the prediction recorded when the model was exported.
# WHY: serving adds JSON parsing, float conversion and a network hop. This is the
# check that proves none of it changed the model's mind. If a row disagrees, the
# serving stack is at fault - the model on disk has not changed since export.
golden_rows = card["sample_batch"]
expected = card["sample_batch_predictions"]

api_predictions = []
for row in golden_rows:
    resp = client.post("/predict", json=dict(zip(FIELD_ORDER, row)))
    api_predictions.append(resp.get_json()["class_id"])

matches = sum(a == e for a, e in zip(api_predictions, expected))
print(f"Golden batch: {matches}/{len(expected)} API answers match the export-time predictions")

for i in range(3):
    print(f"  row {i}: api={card['class_names'][api_predictions[i]]:<12} "
          f"recorded at export={card['class_names'][expected[i]]}")

if matches != len(expected):
    raise AssertionError("The serving layer changed a prediction - do not deploy this.")
print("\nServing layer verified: HTTP in, identical model behaviour out.")


Golden batch: 20/20 API answers match the export-time predictions
  row 0: api=malignant    recorded at export=malignant
  row 1: api=benign       recorded at export=benign
  row 2: api=malignant    recorded at export=malignant

Serving layer verified: HTTP in, identical model behaviour out.


## 6. Production Deployment with Gunicorn

Flask's built-in development server is single-threaded and not safe for production. Gunicorn is a production-grade WSGI server that spawns multiple worker processes.

```bash
# Install
pip install gunicorn

# Run with 4 worker processes (rule of thumb: 2 * CPU cores + 1)
gunicorn flask_app:app --workers 4 --bind 0.0.0.0:5000
```

The request body has one field per feature in *your* card, so build the `curl` from the card rather than copying one from a tutorial — Unit 1's notebook 01 prints a ready-made one for your model.

**Gunicorn vs dev server:**
- Dev server: 1 thread, restarts on code change, never use in production
- Gunicorn: multiple workers, stable, handles concurrent requests

**One caution for ML:** each Gunicorn worker is a separate process that loads its own copy of the artifact. Four workers means four copies of the model in RAM — fine for a 2 KB logistic regression, a real capacity decision for a 500 MB network.

## 7. Summary

In this notebook you:
- Loaded **your portfolio artifact and its card** — the deployment job starts where training ended
- Built a Flask app that reads its required fields from `card["feature_names"]`
- Wrote input validation by hand, returning **400** for bad client input and **500** for server errors
- Tested the entire API in-notebook using `app.test_client()` — no live server needed
- Replayed the card's **golden batch** and proved the HTTP layer did not change a single prediction
- Learned how to run Flask with Gunicorn for production traffic

The Flask serving pattern is: **load artifact + card once at startup → validate input → predict → return JSON with an honest status code**.

**Next:** the same artifact, the same endpoints, served by FastAPI — so you can compare the two frameworks with the model held constant.

## Self-Check (answer before scrolling up)

1. **What does `app.test_client()` give you?** (Hint: think about what you avoided by using it)
2. **What is the difference between Flask's dev server and Gunicorn?** Name at least two differences.
3. **Why return 400 for bad input instead of 500?** What does each status code communicate to the caller?
4. **The golden batch passed. What class of bug does that rule out, and what class does it *not* rule out?**
5. **Why does the app read `card["feature_names"]` instead of a hard-coded list of field names?**

## 📚 References

1. Fielding, R. T. (2000). *Architectural Styles and the Design of Network-based Software Architectures* (Ch. 5: REST). PhD dissertation, University of California, Irvine.
2. Crankshaw, D., Wang, X., Zhou, G., Franklin, M. J., Gonzalez, J. E., & Stoica, I. (2017). *Clipper: A Low-Latency Online Prediction Serving System*. NSDI. <https://arxiv.org/abs/1612.03079>
3. Kreuzberger, D., Kühl, N., & Hirschl, S. (2022). *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*. IEEE Access. <https://arxiv.org/abs/2205.02302>
